In [ ]:
import duckdb
import pandas as pd

# 1. DuckDB instance aur Secret Key configuration
con = duckdb.connect()
HF_TOKEN = "" # Apna valid Hugging Face token yahan lagayein
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# 2. Warehouse paths definition (Official FlyRank Schema)
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f'{REL}/fact_content_daily_performance'

# Feature window ke liye Feb 2026 ka sub-directory data direct scan karenge
FEB_PARTITION = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("⚡ Fetching schemas and top 5 rows from FlyRank Warehouse...\n")

# 3. Daily Performance Table (Feb 2026 Partition) ke sample rows load karna
print("=== FACT_CONTENT_DAILY_PERFORMANCE (Top 5 Rows) ===")
df_fact_sample = con.execute(f"SELECT * FROM {FEB_PARTITION} LIMIT 5").df()
print(df_fact_sample)
print("-" * 50)

# 4. Dim Content Table ke sample rows load karna
print("\n=== DIM_CONTENT (Top 5 Rows) ===")
df_dim_sample = con.execute(f"SELECT * FROM {DIM_CONTENT} LIMIT 5").df()
print(df_dim_sample)


⚡ Fetching schemas and top 5 rows from FlyRank Warehouse...

=== FACT_CONTENT_DAILY_PERFORMANCE (Top 5 Rows) ===
  report_date           client_hash_id           content_hash_id  \
0  2026-02-01  client_e547b89c05043229  content_7995404695ee1ffd   
1  2026-02-01  client_e547b89c05043229  content_1eea820697c3b95a   
2  2026-02-01  client_e547b89c05043229  content_ccbb253f142217c3   
3  2026-02-01  client_e547b89c05043229  content_ae16a6b9cf64c80a   
4  2026-02-01  client_e547b89c05043229  content_acf700633f016e5a   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True                True   
3            True            True                True               False   
4            True            True                True               False   

   gsc_impressi

In [ ]:
# 1. Base Data Matrix banane ki SQL Query
matrix_query = f"""
WITH aggregated_performance AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        SUM(scroll_events) as total_scrolls,
        -- Average Position calculate kar rahe hain (sum_position / impressions)
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_sum_position)::FLOAT / SUM(gsc_impressions))
            ELSE 100
        END as avg_position
    FROM
        {FEB_PARTITION}
    GROUP BY
        content_hash_id
    HAVING
        SUM(gsc_impressions) > 100 -- Data Cleaning: Outliers/Low data ko filter kiya
)

SELECT
    p.content_hash_id,
    p.total_impressions,
    p.total_clicks,
    p.avg_position,
    p.total_scrolls,
    -- Click-Through Rate (CTR) Feature Engineering
    CASE
        WHEN p.total_impressions > 0 THEN (p.total_clicks::FLOAT / p.total_impressions)
        ELSE 0
    END as ctr,
    c.word_count,
    c.content_type
FROM
    aggregated_performance p
LEFT JOIN
    {DIM_CONTENT} c
ON
    p.content_hash_id = c.content_hash_id
"""

print("⏳ Creating Base Data Matrix by joining tables...")
# Query execute karke main matrix DataFrame bana rahe hain
df_matrix = con.execute(matrix_query).df()

print("🎉 Base Data Matrix successfully created!")
print(f"Total rows (Unique Pages) in Matrix: {len(df_matrix)}")
print("\n--- Matrix Sample (Top 5 Rows) ---")
print(df_matrix.head())


⏳ Creating Base Data Matrix by joining tables...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🎉 Base Data Matrix successfully created!
Total rows (Unique Pages) in Matrix: 80124

--- Matrix Sample (Top 5 Rows) ---
            content_hash_id  total_impressions  total_clicks  avg_position  \
0  content_7995404695ee1ffd             1012.0           3.0     28.886364   
1  content_ccbb253f142217c3             1598.0           7.0     17.273466   
2  content_ae16a6b9cf64c80a              861.0           0.0      8.182346   
3  content_acf700633f016e5a              245.0           0.0      8.032653   
4  content_712e44562fed8ff2              306.0           0.0      7.581699   

   total_scrolls       ctr  word_count     content_type  
0            1.0  0.002964        2828  keyword article  
1            0.0  0.004380        2689  keyword article  
2            0.0  0.000000        2928  keyword article  
3            0.0  0.000000        3108  keyword article  
4            0.0  0.000000        2288  keyword article  


In [ ]:
# Chaliye check karte hain ki hamare 80,124 rows mein koi missing value toh nahi hai
print("=== Missing Values (NaN) Check ===")
print(df_matrix.isnull().sum())


=== Missing Values (NaN) Check ===
content_hash_id          0
total_impressions        0
total_clicks             0
avg_position             0
total_scrolls        40491
ctr                      0
word_count           24877
content_type             0
dtype: int64


In [ ]:
# 1. Jin rows mein word_count khali (NaN) hai, unhe nikal dete hain
df_clean = df_matrix.dropna(subset=['word_count']).copy()

# 2. Model ke liye sirf kaam ke numerical features select karte hain
# Hum total_scrolls ko drop kar rahe hain kyunki usme bohot missing data tha
features_to_use = ['total_impressions', 'total_clicks', 'avg_position', 'ctr', 'word_count']

print("=== Course Correction Ke Baad Data Matrix ===")
print(f"Pehle Rows Thin: {len(df_matrix)}")
print(f"Ab Rows Bachi Hain (Clean & Complete): {len(df_clean)}")

# 3. Final cross-check karte hain ki koi aur null toh nahi bacha
print("\n--- Naye Data Frame Mein Missing Values Check ---")
print(df_clean[features_to_use].isnull().sum())


=== Course Correction Ke Baad Data Matrix ===
Pehle Rows Thin: 80124
Ab Rows Bachi Hain (Clean & Complete): 55247

--- Naye Data Frame Mein Missing Values Check ---
total_impressions    0
total_clicks         0
avg_position         0
ctr                  0
word_count           0
dtype: int64


In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Jin columns mein extreme gaps hain, unka Log Transform le rahe hain
df_clean['log_impressions'] = np.log1p(df_clean['total_impressions'])
df_clean['log_clicks'] = np.log1p(df_clean['total_clicks'])
df_clean['log_word_count'] = np.log1p(df_clean['word_count'])

# 2. Final features jo model ko dene hain (Ab isme hum original clicks/impressions ki jagah log wale columns use karenge)
final_features = ['log_impressions', 'log_clicks', 'avg_position', 'ctr', 'log_word_count']

# 3. StandardScaler initialize karna
scaler = StandardScaler()

# 4. Data ko scale (transform) karna
scaled_matrix = scaler.fit_transform(df_clean[final_features])

# 5. Check karne ke liye ise dataframe mein convert karte hain
df_scaled = pd.DataFrame(scaled_matrix, columns=final_features)

print("🎉 Data Preprocessing & Scaling Successful!")
print(f"Scaled Matrix Shape: {df_scaled.shape} (Rows, Features)")
print("\n--- Scaled Data Sample (Top 3 Rows) ---")
print(df_scaled.head(3))


🎉 Data Preprocessing & Scaling Successful!
Scaled Matrix Shape: (55247, 5) (Rows, Features)

--- Scaled Data Sample (Top 3 Rows) ---
   log_impressions  log_clicks  avg_position       ctr  log_word_count
0         0.007735    0.086130      1.837987 -0.010742        0.054985
1         0.345868    0.647691      0.649359  0.289883       -0.077258
2        -0.111837   -1.036991     -0.281155 -0.640087        0.146164


In [ ]:
from sklearn.cluster import KMeans

# 1. K-Means model initialize karna (4 clusters ke sath)
# random_state=42 isliye taaki jab bhi aap code run karein, result hamesha same aaye
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

# 2. Model ko scaled matrix par train karna aur clusters predict karna
print("⏳ Training K-Means Clustering Model on 55,247 pages...")
cluster_labels = kmeans.fit_predict(scaled_matrix)

# 3. Predicted clusters ko hamare main clean dataframe mein add karna
df_clean['cluster'] = cluster_labels

print("🎉 Model Training Complete! Clusters successfully assigned.")
print("\n--- Har Cluster Mein Kitne Pages Aaye (Distribution) ---")
print(df_clean['cluster'].value_counts())


⏳ Training K-Means Clustering Model on 55,247 pages...
🎉 Model Training Complete! Clusters successfully assigned.

--- Har Cluster Mein Kitne Pages Aaye (Distribution) ---
cluster
0    23551
2    18552
1    10496
3     2648
Name: count, dtype: int64


In [ ]:
# Har cluster ka raw performance averages check karte hain
cluster_profile = df_clean.groupby('cluster').agg({
    'total_impressions': 'mean',
    'total_clicks': 'mean',
    'avg_position': 'mean',
    'ctr': 'mean',
    'word_count': 'mean',
    'content_hash_id': 'count'
}).rename(columns={'content_hash_id': 'page_count'}).round(4)

print("=== 📊 CLUSTER PERFORMANCE PROFILES ===")
print(cluster_profile)


=== 📊 CLUSTER PERFORMANCE PROFILES ===
         total_impressions  total_clicks  avg_position     ctr  word_count  \
cluster                                                                      
0                 589.0461        0.8054        8.7969  0.0016   2461.8931   
1                1450.9292        1.4749       23.2024  0.0012   4364.7699   
2                6077.9531       22.0001        7.1753  0.0038   2866.2121   
3                1026.6360       17.9977        7.5464  0.0174    2850.321   

         page_count  
cluster              
0             23551  
1             10496  
2             18552  
3              2648  


In [ ]:
# 1. March 2026 ka path set kar rahe hain
MARCH_PARTITION = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

# 2. March ka aggregate matrix query (Same features ke sath)
march_query = f"""
WITH march_aggregated AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_sum_position)::FLOAT / SUM(gsc_impressions))
            ELSE 100.0
        END as avg_position
    FROM
        {MARCH_PARTITION}
    GROUP BY
        content_hash_id
)

SELECT
    m.content_hash_id,
    m.total_impressions,
    m.total_clicks,
    m.avg_position,
    CASE
        WHEN m.total_impressions > 0 THEN (m.total_clicks::FLOAT / m.total_impressions)
        ELSE 0.0
    END as ctr,
    c.word_count
FROM
    march_aggregated m
LEFT JOIN
    {DIM_CONTENT} c
ON
    m.content_hash_id = c.content_hash_id
WHERE
    c.word_count IS NOT NULL -- Same cleaning rule apply kar rahe hain
"""

print("⏳ March 2026 ka data warehouse se stream ho raha hai...")
df_march = con.execute(march_query).df()

print(f"🎉 March Data Matrix ready! Total unique pages in March: {len(df_march)}")
print(df_march.head())


⏳ March 2026 ka data warehouse se stream ho raha hai...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🎉 March Data Matrix ready! Total unique pages in March: 224008
            content_hash_id  total_impressions  total_clicks  avg_position  \
0  content_d0dff76c889de68f              181.0           0.0      5.171271   
1  content_67741cce996cfafa               46.0           1.0      4.543478   
2  content_2e6360ad20fd7107              899.0           1.0      5.825362   
3  content_ac8663da7484669a               34.0           0.0      5.941176   
4  content_65c50dfe9d87a585             3108.0           0.0      6.953668   

        ctr  word_count  
0  0.000000        2999  
1  0.021739        3057  
2  0.001112        2855  
3  0.000000        3281  
4  0.000000        2779  


In [ ]:
import numpy as np
import pandas as pd

# 1. March ke data par strict filtering (Same data cleaning rule as February)
# Hum sirf complete rows check karenge taaki noise leak na ho
df_march_clean = df_march[(df_march['total_impressions'] > 100) & (df_march['word_count'].notnull())].copy()

print(f"🧹 Noise Saaf Karne Ke Baad March Ki Rows: {len(df_march_clean)}")

# 2. Log Transformation apply karna
df_march_clean['log_impressions'] = np.log1p(df_march_clean['total_impressions'])
df_march_clean['log_clicks'] = np.log1p(df_march_clean['total_clicks'])
df_march_clean['log_word_count'] = np.log1p(df_march_clean['word_count'])

# Same features array jo validation ke liye chahiye
validation_features = ['log_impressions', 'log_clicks', 'avg_position', 'ctr', 'log_word_count']

# 3. February wale trained scaler se March ke filtered data ko transform karna
march_clean_scaled = scaler.transform(df_march_clean[validation_features])

# 4. February wale trained K-Means model se predictions nikalna
df_march_clean['predicted_cluster'] = kmeans.predict(march_clean_scaled)

# 5. Clean March profiles check karna
march_clean_profile = df_march_clean.groupby('predicted_cluster').agg({
    'total_impressions': 'mean',
    'total_clicks': 'mean',
    'avg_position': 'mean',
    'ctr': 'mean',
    'word_count': 'mean',
    'content_hash_id': 'count'
}).rename(columns={'content_hash_id': 'page_count'}).round(4)

print("\n=== 📊 CORRECTED MARCH CLUSTER PERFORMANCE PROFILES ===")
print(march_clean_profile)


🧹 Noise Saaf Karne Ke Baad March Ki Rows: 72342

=== 📊 CORRECTED MARCH CLUSTER PERFORMANCE PROFILES ===
                   total_impressions  total_clicks  avg_position     ctr  \
predicted_cluster                                                          
0                           640.5748        0.8391      8.905200  0.0017   
1                          2129.9238        1.7275     27.866699  0.0012   
2                          7321.9520       24.2367      7.954300  0.0037   
3                          1110.5386       19.5110      7.914700  0.0170   

                   word_count  page_count  
predicted_cluster                          
0                   2534.9291       28192  
1                   3971.6561       15847  
2                   2836.7097       24892  
3                   2806.1996        3411  


In [ ]:
import joblib

# Model aur Scaler ko save kar rahe hain
joblib.dump(kmeans, 'content_clustering_model.pkl')
joblib.dump(scaler, 'content_scaler.pkl')

print("💾 Model aur Scaler safely save ho chuke hain!")


💾 Model aur Scaler safely save ho chuke hain!
